In [1]:
!pip install scanpy
#may need restart after install

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 42.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 198.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.2/174.2 kB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 216.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 229.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 197.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install humanize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 23.2 MB/s eta 0:00:00


In [3]:
!pip install anndata

In [4]:
!pip install git+https://github.com/gillislab/pyMN#egg=pymetaneighbor

  Cloning https://github.com/gillislab/pyMN to /tmp/pip-install-zbpyzonv/pymetaneighbor_3cf020ff9de84f71bb3f0d1750e163dc
  Running command git clone --filter=blob:none --quiet https://github.com/gillislab/pyMN /tmp/pip-install-zbpyzonv/pymetaneighbor_3cf020ff9de84f71bb3f0d1750e163dc
  Resolved https://github.com/gillislab/pyMN to commit 882b54512347be66000f7909271fd8e0cb7def39
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pymetaneighbor: filename=pyMetaNeighbor-0.1.0-py3-none-any.whl size=29125 sha256=30bfaf397c4eecb312f45a5b109b8f9ba5b16a9cec52ce6790d1a31052412306
  Stored in directory: /tmp/pip-ephem-wheel-cache-oj_m153q/wheels/81/4f/2f/ceba31de0b80b66b66844e8aa5fa3604b0f9754c5331b4a80e
  Created wheel for upsetplot: filename=upsetplot-0.9.0-py3-none-any.whl size=24866 sha256=4a8037bfb419461537fdbf1f66c296c4d4430424c071a9e2db2db1f4627e3

In [5]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import scipy.sparse as sp
import time
import os
import gc
import sys
import resource
import time
import datetime
import multiprocessing


In [6]:
base_data_folder = "/sbgenomics/project-files/external_datasets/cellxgene/"

In [7]:
sea_ad = sc.read_h5ad(os.path.join(base_data_folder, 'd3427e8c-c55d-4d4e-b15b-1a8774cd3a4b.h5ad'), backed="r")

n_obs = sea_ad.n_obs

# Randomly select 1/7th of the indices
np.random.seed(42)  # for reproducibility; remove or change seed as needed
subset_size = n_obs // 4
random_indices = np.random.choice(n_obs, size=subset_size, replace=False)
random_indices = np.sort(random_indices)  # sort for efficient h5ad access

# Extract the subset
sea_ad = sc.AnnData(
    sea_ad.raw.X[random_indices, :], 
    obs=sea_ad.obs.iloc[random_indices], 
    var=sea_ad.var
)
sea_ad = sea_ad.to_memory()

In [8]:
sea_ad

AnnData object with n_obs × n_vars = 348900 × 36412
    obs: 'assay_ontology_term_id', 'suspension_type', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'donor_id', 'Neurotypical reference', 'Class', 'Subclass', 'Supertype', 'Age at death', 'Years of education', 'Cognitive status', 'ADNC', 'Braak stage', 'Thal phase', 'CERAD score', 'APOE4 status', 'Lewy body disease pathology', 'LATE-NC stage', 'Microinfarct pathology', 'Specimen ID', 'PMI', 'Number of UMIs', 'Genes detected', 'Fraction mitochrondrial UMIs', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'

In [9]:
ma_2022 = sc.read_h5ad(os.path.join(base_data_folder, '2e0b82ee-5320-437e-b06e-3d2150e70395.only_human.h5ad'))

In [10]:
del ma_2022.raw 

In [11]:
siletti = sc.read_h5ad(os.path.join(base_data_folder, '40a4a15b-f7d8-42cc-a3f1-70963777208c.h5ad'))

In [12]:
del siletti.raw

In [13]:
# Count cells by supercluster and sort by count (descending)
supercluster_counts = pd.DataFrame(siletti.obs['supercluster_term'].value_counts()).reset_index()
supercluster_counts.columns = ['supercluster_term', 'n']
print(supercluster_counts)

                      supercluster_term      n
0        Upper-layer intratelencephalic  11465
1                       MGE interneuron   5268
2         Deep-layer intratelencephalic   4765
3                       CGE interneuron   3951
4                       Oligodendrocyte   1443
5     Deep-layer corticothalamic and 6b   1393
6                             Astrocyte    965
7             LAMP5-LHX6 and Chandelier    575
8             Oligodendrocyte precursor    472
9            Deep-layer near-projecting    408
10                            Microglia    213
11                        Miscellaneous     78
12                             Splatter     23
13                             Vascular     19
14  Committed oligodendrocyte precursor     16
15                           Fibroblast      8
16        Eccentric medium spiny neuron      2
17                    Hippocampal CA1-3      1


In [14]:
# Filter out unwanted cell types
unwanted_terms = ["Eccentric medium spiny neuron", "Hippocampal CA1-3", "Fibroblast", "Splatter"]
siletti = siletti[~siletti.obs['supercluster_term'].isin(unwanted_terms)].copy()

# Count cells by supercluster after filtering
supercluster_counts = pd.DataFrame(siletti.obs['supercluster_term'].value_counts()).reset_index()
supercluster_counts.columns = ['supercluster_term', 'n']
print(supercluster_counts)

# Get dimensions after filtering
print(siletti.shape)



                      supercluster_term      n
0        Upper-layer intratelencephalic  11465
1                       MGE interneuron   5268
2         Deep-layer intratelencephalic   4765
3                       CGE interneuron   3951
4                       Oligodendrocyte   1443
5     Deep-layer corticothalamic and 6b   1393
6                             Astrocyte    965
7             LAMP5-LHX6 and Chandelier    575
8             Oligodendrocyte precursor    472
9            Deep-layer near-projecting    408
10                            Microglia    213
11                        Miscellaneous     78
12                             Vascular     19
13  Committed oligodendrocyte precursor     16
(31031, 59236)


In [15]:
# Create cell_type column from supercluster_term
siletti.obs['study_id'] = "Siletti"
siletti.obs['cell_type'] = siletti.obs['supercluster_term']
siletti.obs['level_1'] = siletti.obs['supercluster_term']
siletti.obs['level_2'] = siletti.obs['cluster_id']
siletti.obs['level_3'] = siletti.obs['subcluster_id']

In [16]:
siletti.obs['study_id'] = siletti.obs['study_id'].to_numpy(dtype="str")
siletti.obs['cell_type'] = siletti.obs['cell_type'].to_numpy(dtype="str")
siletti.obs['level_1'] = siletti.obs['level_1'].to_numpy(dtype="str")
siletti.obs['level_2'] = siletti.obs['level_2'].to_numpy(dtype="str") #cellxgene cell type
siletti.obs['level_3'] = siletti.obs['level_3'].to_numpy(dtype="str") #cellxgene cell type

In [17]:
len(siletti.obs['level_3'].unique())

639

In [18]:
# Count cells by Subclass and sort by count (descending)
subclass_counts = pd.DataFrame(sea_ad.obs['Subclass'].value_counts()).reset_index()
subclass_counts.columns = ['Subclass', 'n']
print(subclass_counts)

# Create cell_type column from Subclass
sea_ad.obs['cell_type'] = sea_ad.obs['Subclass']
sea_ad.obs['level_1'] = sea_ad.obs['Class']
sea_ad.obs['level_2'] = sea_ad.obs['Subclass']
sea_ad.obs['level_3'] = sea_ad.obs['Supertype']
sea_ad.obs['study_id'] = "SEA_AD"

           Subclass      n
0           L2/3 IT  85534
1   Oligodendrocyte  36498
2             Pvalb  29037
3             L5 IT  26086
4               Vip  25002
5         Astrocyte  21718
6             L4 IT  20530
7               Sst  18896
8             Lamp5  13908
9             L6 IT  11791
10    Microglia-PVM  10718
11            L6 CT   7339
12              OPC   7286
13             Sncg   6230
14       Lamp5 Lhx6   5736
15          L5/6 NP   4844
16              L6b   4684
17       Chandelier   3866
18       L6 IT Car3   3403
19             Pax6   2366
20             VLMC   1284
21            L5 ET   1001
22      Endothelial    648
23        Sst Chodl    495


In [19]:
sea_ad.obs['study_id'] = sea_ad.obs['study_id'].to_numpy(dtype="str")
sea_ad.obs['cell_type'] = sea_ad.obs['cell_type'].to_numpy(dtype="str")
sea_ad.obs['level_1'] = sea_ad.obs['level_1'].to_numpy(dtype="str")
sea_ad.obs['level_2'] = sea_ad.obs['level_2'].to_numpy(dtype="str") #cellxgene cell type
sea_ad.obs['level_3'] = sea_ad.obs['level_3'].to_numpy(dtype="str") #cellxgene cell type

In [20]:
# Count cells by subclass and sort by count (descending)
subclass_counts = pd.DataFrame(ma_2022.obs['subclass'].value_counts()).reset_index()
subclass_counts.columns = ['subclass', 'n']
print(subclass_counts)

# Create cell_type column from subclass
ma_2022.obs['cell_type'] = ma_2022.obs['subclass']
ma_2022.obs['level_1'] = ma_2022.obs['class']
ma_2022.obs['level_2'] = ma_2022.obs['subclass']
ma_2022.obs['level_3'] = ma_2022.obs['subtype']
ma_2022.obs['study_id'] = "Ma_2022"

        subclass      n
0          Oligo  45918
1          Astro  23359
2        L2-3 IT  21694
3            OPC   9302
4          Micro   7556
5           Endo   6866
6            SST   6674
7            VIP   6047
8      L3-5 IT-3   5318
9          PVALB   4949
10     L3-5 IT-1   4566
11     L3-5 IT-2   4415
12            PC   4030
13       L6 IT-1   2999
14  ADARB2 KCNG1   2399
15    LAMP5 RELN   2365
16         L6 CT   2150
17           SMC   2106
18        Immune   1779
19           L6B   1739
20       L5-6 NP   1215
21       L6 IT-2   1207
22     PVALB ChC   1008
23    LAMP5 LHX6    871
24          VLMC    680
25       SST HGF    313
26         L5 ET    256
27            RB    239
28       SST NPY    100


In [21]:
ma_2022.obs['study_id'] = ma_2022.obs['study_id'].to_numpy(dtype="str")
ma_2022.obs['cell_type'] = ma_2022.obs['cell_type'].to_numpy(dtype="str")
ma_2022.obs['level_1'] = ma_2022.obs['level_1'].to_numpy(dtype="str")
ma_2022.obs['level_2'] = ma_2022.obs['level_2'].to_numpy(dtype="str") #cellxgene cell type
ma_2022.obs['level_3'] = ma_2022.obs['level_3'].to_numpy(dtype="str") #cellxgene cell type

In [22]:
#GEN_A1 cellxgene

In [23]:
file_ids = [
    '83bb53d2-7eb5-44e1-997e-4875050e104a.h5ad',
    '524a8acc-37e5-44fe-aae9-e9a50e07ed24.h5ad',
    '8a7a61c0-43dc-45cd-b050-21b15f7069ff.h5ad',
    'e1fd8b32-0855-4ab2-b1c4-3e9dae6fd3b2.h5ad'
]

# Load and subsample each file
subsampled_adatas = []

for file_id in file_ids:
    print(f"Loading {file_id}...")
    
    # Read in backed mode
    adata = sc.read_h5ad(os.path.join(base_data_folder, file_id), backed="r")
    
    # Get total number of observations
    n_obs = adata.n_obs
    
    # Randomly select 1/40th of the indices
    subset_size = n_obs // 22
    random_indices = np.random.choice(n_obs, size=subset_size, replace=False)
    random_indices = np.sort(random_indices)  # sort for efficient h5ad access
    
    # Extract the subset
    X_subset = adata.X[random_indices, :]
    var_subset = adata.var
    
    adata_subset = sc.AnnData(
        X_subset,
        obs=adata.obs.iloc[random_indices],
        var=var_subset
    )
    adata_subset = adata_subset.to_memory()
    
    subsampled_adatas.append(adata_subset)
    print(f"  Loaded {subset_size} cells from {n_obs} total")

# Optional: concatenate all subsampled datasets
combined_GEN_A1_cellxgene = sc.concat(subsampled_adatas, merge='same')

Loading 83bb53d2-7eb5-44e1-997e-4875050e104a.h5ad...
  Loaded 188202 cells from 4140453 total
Loading 524a8acc-37e5-44fe-aae9-e9a50e07ed24.h5ad...
  Loaded 67560 cells from 1486324 total
Loading 8a7a61c0-43dc-45cd-b050-21b15f7069ff.h5ad...
  Loaded 31531 cells from 693682 total


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1806: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [24]:
combined_GEN_A1_cellxgene.obs_names_make_unique()

In [25]:
combined_GEN_A1_cellxgene

AnnData object with n_obs × n_vars = 347845 × 34176
    obs: 'Source', 'n_genes', 'n_counts', 'class', 'subclass', 'subtype', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'donor_id', 'suspension_type', 'is_primary_data', 'DLBD_status', 'FTD_status', 'Tauopathy_status', 'Vascular_status', 'Schizophrenia', 'Bipolar_Disorder', 'Parkinson_disease', 'genetic_ancestry', 'disease_ontology_term_id', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'gene_name', 'n_cells', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'

In [26]:
combined_GEN_A1_cellxgene.obs['study_id'] = "cxg_GEN_A1"
combined_GEN_A1_cellxgene.obs['study_id'] = combined_GEN_A1_cellxgene.obs['study_id'].to_numpy(dtype="str")
combined_GEN_A1_cellxgene.obs['cell_type'] = combined_GEN_A1_cellxgene.obs['subclass'].to_numpy(dtype="str")
combined_GEN_A1_cellxgene.obs['level_1'] = combined_GEN_A1_cellxgene.obs['class'].to_numpy(dtype="str")
combined_GEN_A1_cellxgene.obs['level_2'] = combined_GEN_A1_cellxgene.obs['subclass'].to_numpy(dtype="str") 
combined_GEN_A1_cellxgene.obs['level_3'] = combined_GEN_A1_cellxgene.obs['subtype'].to_numpy(dtype="str") 

In [29]:
base_data_folder

'/sbgenomics/project-files/external_datasets/cellxgene/'

In [30]:
#GEN_A3_cellxgene = sc.read_h5ad(os.path.join(base_data_folder, "88ad4023-93b2-4b70-a8a5-b0cf6c438cac.h5ad"), backed="r")

In [31]:
GEN_A3_cellxgene = sc.read_h5ad(os.path.join(base_data_folder, "../../AMPPD_freeze2_7_cx_rareMut_2024_12_16nolayers.h5ad"), backed="r")

In [32]:
GEN_A3_cellxgene

AnnData object with n_obs × n_vars = 2092520 × 17285 backed at '/sbgenomics/project-files/external_datasets/cellxgene/../../AMPPD_freeze2_7_cx_rareMut_2024_12_16nolayers.h5ad'
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'batch', 'rep', 'set', 'SubID_cs', 'HTO_n_cs', 'SubID_vs_w_gt', 'SubID_vs_wo_gt', 'donor_w_gt', 'brain_reg_w_gt', 'donor_wo_gt', 'brain_reg_wo_gt', 'n_counts', 'Channel', 'percent_mito', 'vcfID_w_gt', 'dbl_scr', 'dbl', 'scale', 'leiden_labels', 'Brain_bank', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'leiden_labels_filtCells', 'anno_pegasus_imBr', 'all_clust', 'anno_pegasus_imBr_3lev', 'clust_2levels', 'dissection', 'reg_bb', 'age', 'participant_id', 'sex', 'RIN', 'path_braak_lb', 'diagnosis_final', 'leide

In [34]:
# Find indices where tissue contains "cortex" (case-insensitive)
#cortex_mask = GEN_A3_cellxgene.obs['tissue'].str.contains('cortex', case=False, na=False)
cortex_mask = GEN_A3_cellxgene.obs['brain_reg_w_gt'].isin(['PMC', 'PFC', 'PVC'])

cortex_indices = np.where(cortex_mask)[0]

print(f"Found {len(cortex_indices)} cells with 'cortex' in tissue out of {GEN_A3_cellxgene.n_obs} total")

# Randomly select some of the cortex cells
subset_size = len(cortex_indices) // 4
random_subset = np.random.choice(cortex_indices, size=subset_size, replace=False)
random_subset = np.sort(random_subset)  # sort for efficient h5ad access

print(f"Subsampling to {subset_size} cells")

# Extract the subset
X_subset = GEN_A3_cellxgene.X[random_subset, :]
var_subset = GEN_A3_cellxgene.var

GEN_A3_cellxgene = sc.AnnData(
    X_subset,
    obs=GEN_A3_cellxgene.obs.iloc[random_subset],
    var=var_subset
)
GEN_A3_cellxgene = GEN_A3_cellxgene.to_memory()

print(f"Loaded {GEN_A3_cellxgene.n_obs} cortex cells into memory")

Found 1405334 cells with 'cortex' in tissue out of 2092520 total
Subsampling to 351333 cells (1/7th of cortex cells)
Loaded 351333 cortex cells into memory


In [26]:
#filter out the cell types that don't fit in cortex

In [36]:
GEN_A3_cellxgene.obs["derived_subclass2_Dec2024"].value_counts() #was derived_class2

derived_subclass2_Dec2024
Oligo                 155299
EN_L2_3_IT             33958
Astro_protoplasmic     31733
OPC                    16329
EN_L4_IT               14896
EN_L3_5_IT_3           10832
IN_VIP                 10369
EN_L3_5_IT_2           10077
IN_PVALB                9688
Astro_fibrous           9610
Mg_Adapt                8593
IN_SST                  8094
Mg_Homeo                4482
EN_L6_IT_1              3432
IN_LAMP5_RELN           3105
EN_L6_CT                3058
IN_ADARB2               3048
EN_L6B                  1920
IN_PVALB_CHC            1916
EN_L5_6_NP              1780
IN_LAMP5_LHX6           1657
Endo                    1656
EN_L6_IT_2              1519
VLMC                    1251
PC                      1103
Mg_DAM                   660
PVM                      411
Mg_Prolif                266
SMC                      257
Adaptive                 235
EN_L5_ET                  64
CD11a+                    35
Ependymal                  0
Chol_PPN_CHRNA3  

In [37]:
# build mask of cells to KEEP - doesn't do much outside of cxgene
keep_mask = ~GEN_A3_cellxgene.obs["derived_subclass2_Dec2024"].isin(["Ependymal", "GPI_Neu", "DMNX_Neu"])

# optional: check counts before/after
print("Total cells before:", GEN_A3_cellxgene.n_obs)
print("Cells to remove:", (~keep_mask).sum())

# subset AnnData in place (new object)
GEN_A3_cellxgene = GEN_A3_cellxgene[keep_mask].copy()

print("Total cells after:", GEN_A3_cellxgene.n_obs)

Total cells before: 351333
Cells to remove: 0
Total cells after: 351333


In [39]:
GEN_A3_cellxgene

AnnData object with n_obs × n_vars = 351333 × 17285
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'batch', 'rep', 'set', 'SubID_cs', 'HTO_n_cs', 'SubID_vs_w_gt', 'SubID_vs_wo_gt', 'donor_w_gt', 'brain_reg_w_gt', 'donor_wo_gt', 'brain_reg_wo_gt', 'n_counts', 'Channel', 'percent_mito', 'vcfID_w_gt', 'dbl_scr', 'dbl', 'scale', 'leiden_labels', 'Brain_bank', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'leiden_labels_filtCells', 'anno_pegasus_imBr', 'all_clust', 'anno_pegasus_imBr_3lev', 'clust_2levels', 'dissection', 'reg_bb', 'age', 'participant_id', 'sex', 'RIN', 'path_braak_lb', 'diagnosis_final', 'leiden_labels_subclass_res1', 'leiden_labels_subtype_res1', 'anno_peg_imBr3lev_consol', 'S_score', 'G2M_score', 'phase', 'Source'

In [40]:
GEN_A3_cellxgene.obs['study_id'] = "freeze2_GEN_A3_ctx"
GEN_A3_cellxgene.obs['study_id'] = GEN_A3_cellxgene.obs['study_id'].to_numpy(dtype="str")
GEN_A3_cellxgene.obs['cell_type'] = GEN_A3_cellxgene.obs['derived_subclass2_Dec2024'].to_numpy(dtype="str")
GEN_A3_cellxgene.obs['level_1'] = GEN_A3_cellxgene.obs['derived_class2_Dec2024'].to_numpy(dtype="str")
GEN_A3_cellxgene.obs['level_2'] = GEN_A3_cellxgene.obs['derived_subclass2_Dec2024'].to_numpy(dtype="str") #cellxgene cell type
GEN_A3_cellxgene.obs['level_3'] = GEN_A3_cellxgene.obs['derived_subtype2_Dec2024'].to_numpy(dtype="str") #cellxgene cell type

In [48]:
GEN_A3_cellxgene.var = GEN_A3_cellxgene.var.set_index('gene_id')

In [41]:
GEN_A3_cellxgene

AnnData object with n_obs × n_vars = 351333 × 17285
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'batch', 'rep', 'set', 'SubID_cs', 'HTO_n_cs', 'SubID_vs_w_gt', 'SubID_vs_wo_gt', 'donor_w_gt', 'brain_reg_w_gt', 'donor_wo_gt', 'brain_reg_wo_gt', 'n_counts', 'Channel', 'percent_mito', 'vcfID_w_gt', 'dbl_scr', 'dbl', 'scale', 'leiden_labels', 'Brain_bank', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'leiden_labels_filtCells', 'anno_pegasus_imBr', 'all_clust', 'anno_pegasus_imBr_3lev', 'clust_2levels', 'dissection', 'reg_bb', 'age', 'participant_id', 'sex', 'RIN', 'path_braak_lb', 'diagnosis_final', 'leiden_labels_subclass_res1', 'leiden_labels_subtype_res1', 'anno_peg_imBr3lev_consol', 'S_score', 'G2M_score', 'phase', 'Source'

In [49]:
sea_ad.var

,feature_is_filtered,feature_name,feature_reference,feature_biotype,feature_length,feature_type
ENSG00000000003,False,TSPAN6,NCBITaxon:9606,gene,2396,protein_coding
ENSG00000000005,False,TNMD,NCBITaxon:9606,gene,873,protein_coding
ENSG00000000419,False,DPM1,NCBITaxon:9606,gene,1262,protein_coding
ENSG00000000457,False,SCYL3,NCBITaxon:9606,gene,2916,protein_coding
ENSG00000000460,False,C1orf112,NCBITaxon:9606,gene,2661,protein_coding
...,...,...,...,...,...,...
ENSG00000288701,False,PRRC2B,NCBITaxon:9606,gene,5158,protein_coding
ENSG00000288702,False,UGT1A3,NCBITaxon:9606,gene,2431,protein_coding
ENSG00000288705,False,UGT1A5,NCBITaxon:9606,gene,2431,protein_coding
ENSG00000288709,False,F8A2,NCBITaxon:9606,gene,1707,protein_coding


In [50]:
GEN_A3_cellxgene.var

,gene_name,chr,gene_type,n_cells,percent_cells,robust,highly_variable_features,mean,var,hvf_loess,hvf_rank,ribosomal,mitochondrial,protein_coding,robust_protein_coding,bins,gene_chrom
gene_id,,,,,,,,,,,,,,,,,
ENSG00000186827,TNFRSF4,1,protein_coding,7846,0.363148,True,False,0.007052,0.015475,0.015640,32161,False,False,True,True,"(0.00684, 0.00813]",1
ENSG00000186891,TNFRSF18,1,protein_coding,9000,0.416560,True,False,0.007644,0.015920,0.016944,43309,False,False,True,True,"(0.00684, 0.00813]",1
ENSG00000160072,ATAD3B,1,protein_coding,340777,15.772689,True,False,0.346708,0.714936,0.769945,55381,False,False,True,True,"(0.326, 0.382]",1
ENSG00000041988,THAP3,1,protein_coding,209923,9.716179,True,False,0.216086,0.479527,0.486180,40148,False,False,True,True,"(0.202, 0.237]",1
ENSG00000142611,PRDM16,1,protein_coding,283396,13.116839,True,True,0.413035,1.224811,0.896464,1349,False,False,True,True,"(0.382, 0.448]",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000177151,OR2T35,1,protein_coding,8,0.000370,False,False,0.000005,0.000008,0.000000,-1,False,False,True,False,"(-0.000999417, 1.08e-05]",1
ENSG00000196944,OR2T4,1,protein_coding,8,0.000370,False,False,0.000006,0.000009,0.000000,-1,False,False,True,False,"(-0.000999417, 1.08e-05]",1
ENSG00000255398,HCAR3,12,protein_coding,6,0.000278,False,False,0.000007,0.000018,0.000000,-1,False,False,True,False,"(-0.000999417, 1.08e-05]",12


In [51]:
merged=ad.concat([ma_2022, sea_ad, siletti, GEN_A3_cellxgene, combined_GEN_A1_cellxgene], join="inner")

In [52]:
merged

AnnData object with n_obs × n_vars = 1251229 × 15112
    obs: 'cell_type', 'sex', 'level_1', 'level_2', 'level_3', 'study_id'

In [53]:
merged.obs['study_id'] = merged.obs['study_id'].to_numpy(dtype="str")
merged.obs['cell_type'] = merged.obs['cell_type'].to_numpy(dtype="str")
merged.obs['level_1'] = merged.obs['level_1'].to_numpy(dtype="str")
merged.obs['level_2'] = merged.obs['level_2'].to_numpy(dtype="str") 
merged.obs['level_3'] = merged.obs['level_3'].to_numpy(dtype="str") 

In [54]:
merged.write("/sbgenomics/output-files/cortex_references_human_downsampled.Jan2026.h5ad")

In [55]:
print("Done!")

Done!


In [56]:
merged_counts = pd.DataFrame(merged.obs['study_id'].value_counts()).reset_index()
merged_counts.columns = ['study_term', 'n']
print(merged_counts)

           study_term       n
0  freeze2_GEN_A3_ctx  351333
1              SEA_AD  348900
2          cxg_GEN_A1  347845
3             Ma_2022  172120
4             Siletti   31031


In [57]:
merged_counts = pd.DataFrame(
    merged.obs.groupby('study_id')['cell_type'].nunique()
).reset_index()
merged_counts.columns = ['study_term', 'n']
print(merged_counts)

           study_term   n
0             Ma_2022  29
1              SEA_AD  24
2             Siletti  14
3          cxg_GEN_A1  27
4  freeze2_GEN_A3_ctx  32


/tmp/ipykernel_237/3830452245.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged.obs.groupby('study_id')['cell_type'].nunique()


In [58]:
merged

AnnData object with n_obs × n_vars = 1251229 × 15112
    obs: 'cell_type', 'sex', 'level_1', 'level_2', 'level_3', 'study_id'